# Newsgroup Document Classification — Mistral 7B + LoRA (CUDA)

Two-phase fine-tuning pipeline for 20 Newsgroups classification.

This notebook imports and runs the existing Python modules directly — no code is duplicated here.

- **Phase 1**: 5 well-separated categories (3 epochs) — pipeline validation
- **Phase 2**: All 20 categories (5 epochs) — full classification challenge

Requires: NVIDIA GPU with >= 24 GB VRAM

## 1. Setup

In [ ]:
!pip install torch>=2.1.0 transformers>=4.36.0 accelerate>=0.25.0 peft>=0.8.0 \
    safetensors>=0.4.0 scikit-learn>=1.3.1 seaborn>=0.12.0 matplotlib>=3.7.0 numpy>=1.24.0

In [ ]:
import logging
import sys

import torch

# Ensure log output is visible in notebook cells
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Allow importing prepare_data from the parent code/ directory
sys.path.insert(0, "..")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected.")

## 2. Prepare Data

Downloads 20 Newsgroups, tokenizes, chunks, and writes JSON files to `../data/`.

Only needs to run once — skip this cell on subsequent runs.

In [ ]:
# prepare_data.py must run from the code/ directory so it writes to code/data/
!(cd .. && python prepare_data.py)

## 3. Phase 1 — 5-Class Training

In [ ]:
import config
import train

train.set_seed(config.RANDOM_SEED)

phase1_data = train.load_phase_data(config.PHASE1_DATA)
model, tokenizer = train.build_model(num_classes=phase1_data["num_classes"])
train_loader, val_loader, _ = train.create_dataloaders(phase1_data, tokenizer.pad_token_id)

phase1_results = train.train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_doc_labels=phase1_data["val_doc_labels"],
    num_epochs=config.PHASE1_EPOCHS,
    checkpoint_dir="phase1",
    phase_name="Phase 1 (5-class)",
)

del model, train_loader, val_loader
torch.cuda.empty_cache()

print(f"\nPhase 1 val accuracies: {[f'{a*100:.1f}%' for a in phase1_results['epoch_accs']]}")

## 4. Phase 2 — 20-Class Training

In [ ]:
train.set_seed(config.RANDOM_SEED)

phase2_data = train.load_phase_data(config.PHASE2_DATA)
model, tokenizer = train.build_model(num_classes=phase2_data["num_classes"])
train_loader, val_loader, _ = train.create_dataloaders(phase2_data, tokenizer.pad_token_id)

phase2_results = train.train_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_doc_labels=phase2_data["val_doc_labels"],
    num_epochs=config.PHASE2_EPOCHS,
    checkpoint_dir="phase2",
    phase_name="Phase 2 (20-class)",
)

del model, train_loader, val_loader
torch.cuda.empty_cache()

print(f"\nPhase 2 val accuracies: {[f'{a*100:.1f}%' for a in phase2_results['epoch_accs']]}")

## 5. Evaluation

Full test-set evaluation with classification report, confusion matrix, and topic-group breakdown.

Set `EVAL_PHASE` to `1` or `2`.

In [ ]:
import json
from functools import partial

from torch.utils.data import DataLoader
from transformers import AutoTokenizer

import evaluate as eval_module

EVAL_PHASE = 2  # Change to 1 for Phase 1

device = "cuda" if torch.cuda.is_available() else "cpu"

if EVAL_PHASE == 1:
    data_path = config.PHASE1_DATA
    checkpoint_dir = config.CHECKPOINT_DIR / "phase1"
    phase_name = "Phase 1 (5-class)"
else:
    data_path = config.PHASE2_DATA
    checkpoint_dir = config.CHECKPOINT_DIR / "phase2"
    phase_name = "Phase 2 (20-class)"

with open(data_path, "r") as f:
    eval_data = json.load(f)

label_names = eval_data["label_names"]
num_classes = eval_data["num_classes"]

eval_tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token
    eval_tokenizer.pad_token_id = eval_tokenizer.eos_token_id

test_dataset = eval_module.ChunkDataset(
    eval_data["test_chunks"], eval_data["test_labels"], doc_ids=eval_data["test_doc_ids"]
)
collate_fn = partial(eval_module.dynamic_padding_collate, pad_token_id=eval_tokenizer.pad_token_id)
test_loader = DataLoader(
    test_dataset, batch_size=config.BATCH_SIZE,
    shuffle=False, num_workers=0, drop_last=False, collate_fn=collate_fn,
)

eval_model = eval_module.load_trained_model(num_classes, checkpoint_dir, device)

print(f"\n{'=' * 70}")
print(f"{phase_name} — Test Set Evaluation")
print(f"{'=' * 70}")

all_preds, all_labels = eval_module.evaluate_with_aggregation(
    eval_model, test_loader, eval_data["test_doc_labels"],
    eval_data["test_num_docs"], label_names, device,
)

In [ ]:
cm_save_path = str(checkpoint_dir / "confusion_matrix.png")
eval_module.plot_confusion_matrix(
    all_labels, all_preds, label_names,
    title=f"{phase_name} — Confusion Matrix", save_path=cm_save_path,
)

if num_classes > 5:
    eval_module.print_accuracy_by_group(all_labels, all_preds, label_names)
    eval_module.print_top_confused_pairs(all_labels, all_preds, label_names)

## 6. Inference Demo

In [ ]:
import inference

print(f"{'=' * 70}")
print(f"Demo: {phase_name} Document Classification")
print(f"{'=' * 70}\n")

for text in inference.DEMO_TEXTS:
    result = inference.classify_document(text, eval_model, eval_tokenizer, device, label_names)
    display_text = text[:70] + "..." if len(text) > 70 else text
    print(f"Text: {display_text}")
    print(f"  Prediction: {result['prediction']} ({result['confidence']:.1f}%)")
    print(f"  Top 3: {', '.join(f'{n} ({c:.1f}%)' for n, c in result['top3'])}")
    print(f"  [{result['num_tokens']} tokens, {result['num_chunks']} chunk(s)]")
    print()

### Try your own text

In [ ]:
custom_text = "Enter your own text here to classify it."

result = inference.classify_document(custom_text, eval_model, eval_tokenizer, device, label_names)
print(f"Prediction: {result['prediction']} ({result['confidence']:.1f}%)")
print(f"Top 3: {', '.join(f'{n} ({c:.1f}%)' for n, c in result['top3'])}")
print(f"[{result['num_tokens']} tokens, {result['num_chunks']} chunk(s)]")